### Convert model to TFLite INT8

#### Pipeline Overview
* best.py (float32):
    * step 1: Magnitude pruning -- structured sparsity on Conv layers // target: 20% 
    * step 2: Fine-tune (optional, 5 epochs) -- recover accuracy after pruning
    * step 3: Export to TFLite float16 -> best_fp16.tflite
    * step 4: Post-Training INT8 Quantization -> best_int8.tflite

* **requirement:** 
    * INT8 mAP@50 must stay within 3 point of float32 baseline (>= 0.907)
    * Model size must be <= 10 MB
    * Single-frame latency <= 5s on Pi 3B

In [68]:
import os
import sys
import copy
import random
import time
import shutil
import warnings
import subprocess
from pathlib import Path
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

warnings.filterwarnings('ignore')

In [119]:
# Env variable 

# Project paths
PROJECT_ROOT = Path.cwd().parent
YOLOV5_DIR= PROJECT_ROOT / 'model' / 'yolov5'
WEIGHTS_PT = PROJECT_ROOT / 'model' / 'train' / 'edge_cctv_v2' / 'weights' / 'best.pt'
TRAIN_WEIGHTS_DIR = WEIGHTS_PT.parent   # model/train/edge_cctv_v2/weights/ — already exists
SAVE_PLOTS_DIR = PROJECT_ROOT / 'plots'

EXPORT_DIR= PROJECT_ROOT / 'model' / 'train' / 'edge_cctv_v2' / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODEL_DIR = EXPORT_DIR / 'saved_model'
TEST_IMGS_DIR = PROJECT_ROOT / 'data' / 'splits' / 'test' / 'images'
TFLITE_INT8 = EXPORT_DIR / 'best_int8.tflite'
DATA_YAML = PROJECT_ROOT / 'data' / 'data.yaml'
TRAIN_IMGS = PROJECT_ROOT / 'data' / 'splits' /'train' / 'images'


PRUNE_AMOUNT = 0.20

# Add yolov5 to python path so we can import its utils
sys.path.insert(0, str(YOLOV5_DIR))

print('Environment ready')
print(f'PyTorch : {torch.__version__}')
print(f'TF: {tf.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU (export only)"}')
print(f'weights: {WEIGHTS_PT.exists()}')

Environment ready
PyTorch : 2.11.0+cpu
TF: 2.21.0
CUDA: False
Device: CPU (export only)
weights: True


#### Step 1: Load the best line model (float32)

In [109]:
model = torch.hub.load(
    str(YOLOV5_DIR), 'custom',
    path=str(WEIGHTS_PT), source='local',
    force_reload=False, verbose=False,
)
model = model.float().eval()

total = sum(p.numel() for p in model.parameters())
nz = sum(p.nonzero().size(0) for p in model.parameters())
print(f'Parameters : {total:,}  |  sparsity: {100*(1-nz/total):.1f}%')

YOLOv5  v7.0-484-g70b964b6 Python-3.12.13 torch-2.11.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 1760518 parameters, 0 gradients, 4.1 GFLOPs
Adding AutoShape... 


Parameters : 1,760,518  |  sparsity: 0.0%


#### step 2: Magnitude-based structured pruning

* **strategy**:
    * Unstructured pruning zeroes out the lowest-magnitude weights.  
    * After calling `prune.remove()` the mask is made permanent - the sparse weights are baked into the checkpoint.  
    * **Target sparsity: 20%** - conservative choice that typically costs < 1 mAP point on YOLOv5n.  

In [110]:
model_pruned = copy.deepcopy(model)
pruned_layers = 0
for name, module in model_pruned.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.l1_unstructured(module, name='weight', amount=PRUNE_AMOUNT)
        prune.remove(module, 'weight')
        pruned_layers += 1

nz2 = sum(p.nonzero().size(0) for p in model_pruned.parameters())
tot = sum(p.numel() for p in model_pruned.parameters())
print(f'Pruned {pruned_layers} Conv2d layers')
print(f'Sparsity after: {100*(1-nz2/tot):.1f}%')


Pruned 60 Conv2d layers
Sparsity after: 19.9%


In [111]:
# inference check inference (pruned model)
dummy = torch.zeros(1, 3, 640, 640)   # YOLOv5 default input size
with torch.no_grad():
    out = model_pruned(dummy)

print(f'Forward pass OK - output shape: {out[0].shape}')

Forward pass OK - output shape: torch.Size([25200, 6])


### Step 3: Export best.pt -> TF SavedModel

In [112]:
# Remove stale SavedModel if re-running
if SAVED_MODEL_DIR.exists():
    shutil.rmtree(SAVED_MODEL_DIR)
    print('Removed stale SavedModel')

print('Exporting best.pt -> SavedModel')
result = subprocess.run(
    ['python', str(YOLOV5_DIR/'export.py'),
     '--weights', str(WEIGHTS_PT),
     '--include', 'saved_model',
     '--img', '640', '--batch-size', '1',
     '--device', 'cpu', '--simplify', '--opset', '12'],
    capture_output=True, text=True,
    encoding='utf-8', errors='replace',
    cwd=str(YOLOV5_DIR)
)

if result.returncode == 0:
    # export.py writes next to --weights: TRAIN_WEIGHTS_DIR/best_saved_model/
    src_candidates = list(TRAIN_WEIGHTS_DIR.glob('*saved_model*'))
    if src_candidates:
        shutil.move(str(src_candidates[0]), str(SAVED_MODEL_DIR))
        print(f'SavedModel -> {SAVED_MODEL_DIR}')
    else:
        print('SavedModel not found next to weights - check TRAIN_WEIGHTS_DIR')
else:
    print('Export failed:')
    print(result.stderr[-3000:])

Removed stale SavedModel
Exporting best.pt -> SavedModel
SavedModel -> c:\Users\ASUS\Desktop\EdgeIA_PFE\model\train\edge_cctv_v2\export\saved_model


#### Step 4: Verify SavedModel I/O signature

In [113]:
loaded = tf.saved_model.load(str(SAVED_MODEL_DIR))
infer  = loaded.signatures['serving_default']

print('SavedModel inputs:')
for k, v in infer.structured_input_signature[1].items():
    print(f'{k}: shape={v.shape}  dtype={v.dtype}')

print('SavedModel outputs:')
for k, v in infer.structured_outputs.items():
    print(f'{k}: shape={v.shape}  dtype={v.dtype}')

SavedModel inputs:
x: shape=(1, 640, 640, 3)  dtype=<dtype: 'float32'>
SavedModel outputs:
output_0: shape=(1, 25200, 6)  dtype=<dtype: 'float32'>


### Step 5: Representative dataset for INT8 calibration

In [114]:
N_CALIB  = 200
IMG_SIZE = 640

all_imgs   = list(TRAIN_IMGS.glob('*.jpg')) + list(TRAIN_IMGS.glob('*.png'))
random.seed(42)
calib_imgs = random.sample(all_imgs, min(N_CALIB, len(all_imgs)))
print(f'Calibration set: {len(calib_imgs)} images')

def representative_data_gen():
    """Yields float32 [0,1] images — matches SavedModel input dtype."""
    for img_path in calib_imgs:
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        yield [arr[np.newaxis]]   # shape: (1, 640, 640, 3)

Calibration set: 200 images


### Step 6: INT Post-training Quantization

In [115]:
# Remove stale INT8 model before converting
if TFLITE_INT8.exists():
    TFLITE_INT8.unlink()
    print('Deleted stale best_int8.tflite')

print('Converting -> TFLite INT8 (PTQ)')
t0 = time.time()

converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,    # float32 fallback for Detect head ops
]
# NOTE: do NOT set inference_input_type / inference_output_type
# Forcing uint8 I/O on YOLOv5 makes the converter set output scale=0 → all scores = 0

tflite_model = converter.convert()

with open(TFLITE_INT8, 'wb') as f:
    f.write(tflite_model)

size_mb = TFLITE_INT8.stat().st_size / 1e6
print(f'\nINT8 model saved -> {TFLITE_INT8}')
print(f'Calibration time: {time.time()-t0:.0f}s')
print(f'Model size: {size_mb:.2f} MB')
print(f'Size constraint: {"PASS" if size_mb <= 10 else "FAIL"} (≤ 10 MB)')

Deleted stale best_int8.tflite
Converting -> TFLite INT8 (PTQ)



INT8 model saved -> c:\Users\ASUS\Desktop\EdgeIA_PFE\model\train\edge_cctv_v2\export\best_int8.tflite
Calibration time: 56s
Model size: 2.00 MB
Size constraint: PASS (≤ 10 MB)


### Step 7: Load interpreter & inspect tensors

In [116]:
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_INT8))
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

print(f'Input: shape={inp["shape"]}  dtype={inp["dtype"]}  scale={inp["quantization"][0]:.4f}')
print(f'Output: shape={out["shape"]}  dtype={out["dtype"]}  scale={out["quantization"][0]:.6f}')

assert inp['dtype'] == np.float32, f'Expected float32 input, got {inp["dtype"]}'
print('\nTensor dtypes confirmed - float32 I/O with INT8 internal weights')

Input: shape=[  1 640 640   3]  dtype=<class 'numpy.float32'>  scale=0.0000
Output: shape=[    1 25200     6]  dtype=<class 'numpy.float32'>  scale=0.000000

Tensor dtypes confirmed - float32 I/O with INT8 internal weights


### Step 8: Dummy inference - latency benchmark

In [117]:
dummy = np.random.rand(1, 640, 640, 3).astype(np.float32)
interpreter.set_tensor(inp['index'], dummy)

# Warm-up
interpreter.invoke()

# Timed run
t0 = time.time()
interpreter.invoke()
latency_ms = (time.time() - t0) * 1000

raw = interpreter.get_tensor(out['index'])  # (1, 25200, 6)
print(f'Dummy inference OK')
print(f'Output shape: {raw.shape}')
print(f'Latency: {latency_ms:.1f} ms (host CPU)')
print(f'Est. Pi 3B: {latency_ms*10:.0f}–{latency_ms*15:.0f} ms')

Dummy inference OK
Output shape: (1, 25200, 6)
Latency: 57.0 ms (host CPU)
Est. Pi 3B: 570–855 ms


### Step 9: Real image inference with NMS

In [122]:
sample_path = next(TEST_IMGS_DIR.glob('*.jpg'), None) or next(TEST_IMGS_DIR.glob('*.png'), None)
if sample_path is None:
    print('No test images found')
else:
    # Preprocess
    img_orig = Image.open(sample_path).convert('RGB')
    img_resized = img_orig.resize((640, 640), Image.BILINEAR)
    img_arr = np.array(img_resized, dtype=np.float32) / 255.0  # [0,1]
    img_tensor  = img_arr[np.newaxis]                               # (1,640,640,3)

    # Inference
    interpreter.set_tensor(inp['index'], img_tensor)
    t0 = time.time()
    interpreter.invoke()
    latency_ms = (time.time() - t0) * 1000

    preds = interpreter.get_tensor(out['index'])[0]   # (25200, 6)

    # Score: objectness only (single-class model)
    objectness = preds[:, 4]

    # Adaptive threshold
    # Use 50% of max score as threshold (adapts to INT8 score compression)
    # with a floor of 0.10 and a ceil of 0.25
    CONF_THRESH = float(np.clip(objectness.max() * 0.5, 0.10, 0.25))
    print(f'Max objectness: {objectness.max():.4f}')
    print(f'Adaptive thresh: {CONF_THRESH:.4f}')

    mask = objectness > CONF_THRESH
    boxes_filtered = preds[mask]
    scores_filtered = objectness[mask]

    # Simple greedy NMS
    def xywh2xyxy(boxes):
        cx, cy, w, h = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
        return np.stack([cx-w/2, cy-h/2, cx+w/2, cy+h/2], axis=1)

    def nms(boxes_xywh, scores, iou_thresh=0.45):
        if len(boxes_xywh) == 0:
            return []
        xyxy   = xywh2xyxy(boxes_xywh) * 640   # pixel coords
        order  = scores.argsort()[::-1]
        keep   = []
        while len(order):
            i = order[0]; keep.append(i)
            if len(order) == 1: break
            rest = order[1:]
            xx1 = np.maximum(xyxy[i,0], xyxy[rest,0])
            yy1 = np.maximum(xyxy[i,1], xyxy[rest,1])
            xx2 = np.minimum(xyxy[i,2], xyxy[rest,2])
            yy2 = np.minimum(xyxy[i,3], xyxy[rest,3])
            inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
            area_i = (xyxy[i,2]-xyxy[i,0])*(xyxy[i,3]-xyxy[i,1])
            area_rest = (xyxy[rest,2]-xyxy[rest,0])*(xyxy[rest,3]-xyxy[rest,1])
            iou = inter / (area_i + area_rest - inter + 1e-6)
            order = rest[iou < iou_thresh]
        return keep

    keep_idx = nms(boxes_filtered, scores_filtered)
    final_boxes = boxes_filtered[keep_idx]
    final_scores = scores_filtered[keep_idx]

    print(f'\nReal image inference')
    print(f'Image: {sample_path.name}')
    print(f'Latency: {latency_ms:.1f} ms')
    print(f'Pre-NMS dets: {len(boxes_filtered)}')
    print(f'Post-NMS dets: {len(final_boxes)}')
    print(f'Score range: {objectness.min():.4f} – {objectness.max():.4f}')

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(img_resized)
    for i, box in enumerate(final_boxes):
        cx, cy, w, h = box[:4]
        x1 = (cx - w/2) * 640;  y1 = (cy - h/2) * 640
        rect = patches.Rectangle((x1,y1), w*640, h*640,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        axes[0].add_patch(rect)
        axes[0].text(x1, y1-4, f'{final_scores[i]:.2f}',
                     color='lime', fontsize=8, fontweight='bold')
    axes[0].set_title(f'INT8 TFLite - {len(final_boxes)} persons | {latency_ms:.0f}ms')
    axes[0].axis('off')

    axes[1].hist(objectness, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[1].axvline(CONF_THRESH, color='red', linestyle='--',
                    label=f'threshold={CONF_THRESH:.3f}')
    axes[1].set_xlabel('Objectness score')
    axes[1].set_ylabel('Anchor count')
    axes[1].set_title('Score distribution - 25200 anchors')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(SAVE_PLOTS_DIR / 'sample_int8_detection.png', dpi=120)
    plt.show()
    print(f'   Preview -> {SAVE_PLOTS_DIR / "sample_int8_detection.png"}')

Max objectness: 0.1180
Adaptive thresh: 0.1000

Real image inference
Image: GX010023_frame_00002_jpg.rf.Ff2mG9xiy9lAKIMfYp3W.jpg
Latency: 55.4 ms
Pre-NMS dets: 2
Post-NMS dets: 2
Score range: 0.0000 – 0.1180
   Preview -> c:\Users\ASUS\Desktop\EdgeIA_PFE\plots\sample_int8_detection.png


### Step 10: Float32 baseline comparison

In [123]:
if sample_path:
    results = model(str(sample_path), size=640)
    results.print()
    results.save(save_dir=str(EXPORT_DIR / 'baseline_detection'))
    print(f'\nFloat32 detections: {len(results.xyxy[0])}')
    print('Compare with INT8 post-NMS above to assess accuracy retention.')

image 1/1: 900x3250 14 persons
Speed: 40.7ms pre-process, 52.3ms inference, 37.9ms NMS per image at shape (1, 3, 192, 640)
Saved 1 image to c:\Users\ASUS\Desktop\EdgeIA_PFE\model\train\edge_cctv_v2\export\baseline_detection



Float32 detections: 14
Compare with INT8 post-NMS above to assess accuracy retention.


### Step 11: mAP evaluation (val.py)

In [124]:
print('Running val.py on test set...')
result = subprocess.run(
    ['python', str(YOLOV5_DIR/'val.py'),
     '--weights', str(TFLITE_INT8),
     '--data', str(DATA_YAML),
     '--img', '640', '--batch-size', '1',
     '--task', 'test', '--device', 'cpu', '--verbose'],
    capture_output=True, text=True,
    encoding='utf-8', errors='replace',
    cwd=str(YOLOV5_DIR)
)

map50 = None
for line in result.stdout.splitlines():
    # val.py prints: Class  Images  Instances  P  R  mAP50  mAP50-95
    if line.strip().startswith('all'):
        parts = line.split()
        try:    map50 = float(parts[4])
        except: pass

print(result.stdout[-4000:])

BASELINE = 0.937
if map50:
    drop = BASELINE - map50
    ok   = 'PASS' if map50 >= BASELINE - 0.03 else 'FAIL'
    print(f'\nmAP@0.5 float32: {BASELINE}')
    print(f'mAP@0.5 INT8: {map50:.3f}')
    print(f'Drop: {drop:.3f} -> {ok}')
else:
    print('Could not auto-parse mAP - read the output above manually.')

Running val.py on test set...

Could not auto-parse mAP - read the output above manually.
